<a href="https://colab.research.google.com/github/SerJes03/simuacionProblemaBancario/blob/main/Bancario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# SIMULACIÓN BANCARIA - MODELO M/M/1 POR CAJERO
# Banco de Colombia
# ==============================================================================
# PASO 1: IMPORTAR LIBRERÍAS
# ==============================================================================
!pip install simpy -q
import simpy
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t as t_dist

# ==============================================================================
# PASO 2: CONFIGURACIÓN GENERAL
# ==============================================================================

RANDOM_SEED      = 42
HORAS_OPERACION  = 8
TIEMPO_SIM       = HORAS_OPERACION * 60   # 480 minutos
NUM_REPLICAS     = 10

PROB_RETIRO = 0.70
PROB_PAGO   = 0.30

# ==============================================================================
# PASO 3: DATOS DE ENTRADA
# ==============================================================================

TIPOS_RETIRO = {
    'Rapido':   {'prob': 0.23, 'servicio': 1, 'llegada_ref': 1},
    'Normal':   {'prob': 0.40, 'servicio': 2, 'llegada_ref': 2},
    'Lento':    {'prob': 0.17, 'servicio': 3, 'llegada_ref': 3},
    'MuyLento': {'prob': 0.20, 'servicio': 4, 'llegada_ref': 3},
}

TIPOS_PAGO = {
    'Rapido':   {'prob': 0.10, 'servicio': 3, 'llegada_ref': 1},
    'Normal':   {'prob': 0.20, 'servicio': 3, 'llegada_ref': 2},
    'Lento':    {'prob': 0.30, 'servicio': 5, 'llegada_ref': 3},
    'MuyLento': {'prob': 0.40, 'servicio': 7, 'llegada_ref': 4},
}

# ==============================================================================
# PASO 4: MEDIA DE LLEGADA GLOBAL PONDERADA
# ==============================================================================

def calcular_llegada_ponderada():
    lr = sum(v['llegada_ref'] * v['prob'] for v in TIPOS_RETIRO.values())
    lp = sum(v['llegada_ref'] * v['prob'] for v in TIPOS_PAGO.values())
    return lr * PROB_RETIRO + lp * PROB_PAGO

MEDIA_LLEGADA_GLOBAL = calcular_llegada_ponderada()

# ==============================================================================
# PASO 5: CLASE BANCO - M/M/1 POR CAJERO
# ==============================================================================
# Cada cajero = simpy.Resource(capacity=1) → sistema M/M/1 independiente.
# Los clientes eligen el cajero habilitado para su acción con menor cola.
# ==============================================================================

class Banco:

    def __init__(self, env, escenario):
        self.env       = env
        self.escenario = escenario

        # Cada cajero: Resource(capacity=1) → M/M/1 propio
        # Diccionario: nombre_cajero → Resource
        self.cajeros = {}

        # ------------------------------------------------------------------
        # ESCENARIO 1: 3 CAJEROS MIXTOS
        # Los 3 cajeros atienden Retiro Y Pago → comparten lista de cajeros
        # pero cada uno tiene capacity=1 (M/M/1 individual).
        # ------------------------------------------------------------------
        if escenario == "3M":
            pool = {
                'Cajero_1': simpy.Resource(env, capacity=1),
                'Cajero_2': simpy.Resource(env, capacity=1),
                'Cajero_3': simpy.Resource(env, capacity=1),
            }
            self.cajeros['Retiro'] = pool
            self.cajeros['Pago']   = pool   # mismo pool → todos atienden ambos

        # ------------------------------------------------------------------
        # ESCENARIO 2: 1 CAJERO RETIRO + 2 CAJEROS PAGO
        # ------------------------------------------------------------------
        elif escenario == "1R-2P":
            self.cajeros['Retiro'] = {
                'Cajero_R1': simpy.Resource(env, capacity=1),
            }
            self.cajeros['Pago'] = {
                'Cajero_P1': simpy.Resource(env, capacity=1),
                'Cajero_P2': simpy.Resource(env, capacity=1),
            }

        # ------------------------------------------------------------------
        # ESCENARIO 3: 2 CAJEROS RETIRO + 1 CAJERO PAGO
        # ------------------------------------------------------------------
        elif escenario == "2R-1P":
            self.cajeros['Retiro'] = {
                'Cajero_R1': simpy.Resource(env, capacity=1),
                'Cajero_R2': simpy.Resource(env, capacity=1),
            }
            self.cajeros['Pago'] = {
                'Cajero_P1': simpy.Resource(env, capacity=1),
            }

        # ------------------------------------------------------------------
        # ESCENARIO 4: 4 CAJEROS MIXTOS
        # ------------------------------------------------------------------
        elif escenario == "4M":
            pool = {
                'Cajero_1': simpy.Resource(env, capacity=1),
                'Cajero_2': simpy.Resource(env, capacity=1),
                'Cajero_3': simpy.Resource(env, capacity=1),
                'Cajero_4': simpy.Resource(env, capacity=1),
            }
            self.cajeros['Retiro'] = pool
            self.cajeros['Pago']   = pool

    # ------------------------------------------------------------------
    # Selecciona el cajero disponible con MENOR cola (política M/M/1)
    # ------------------------------------------------------------------
    def get_cajero(self, accion):
        pool = self.cajeros[accion]
        # Elegir cajero con menos clientes esperando
        nombre = min(pool, key=lambda x: len(pool[x].queue))
        return nombre, pool[nombre]

# ==============================================================================
# PASO 6: PROCESO DE CLIENTE
# ==============================================================================

def cliente_sim(env, nombre, banco, accion, subtipo,
                media_servicio, datos, replica_id, escenario):

    llegada = env.now

    # Elegir cajero M/M/1 con menor cola
    nombre_cajero, cajero = banco.get_cajero(accion)

    with cajero.request() as req:
        yield req                                          # esperar turno
        tiempo_espera   = env.now - llegada
        tiempo_servicio = random.expovariate(1.0 / media_servicio)
        yield env.timeout(tiempo_servicio)                 # ser atendido

        datos.append({
            'Escenario':       escenario,
            'Replica':         replica_id,
            'Cliente_ID':      nombre,
            'Accion':          accion,
            'Subtipo':         subtipo,
            'Tiempo_Llegada':  llegada,
            'Tiempo_Espera':   tiempo_espera,
            'Tiempo_Servicio': tiempo_servicio,
            'Tiempo_Total':    tiempo_espera + tiempo_servicio,
            'Cajero_ID':       nombre_cajero,
        })

# ==============================================================================
# PASO 7: GENERADOR DE LLEGADAS
# ==============================================================================

def llegada_clientes(env, banco, datos, replica_id, escenario):
    cliente_id = 0
    while True:
        yield env.timeout(random.expovariate(1.0 / MEDIA_LLEGADA_GLOBAL))
        cliente_id += 1

        # Definir acción
        if random.random() < PROB_RETIRO:
            accion, tabla = 'Retiro', TIPOS_RETIRO
        else:
            accion, tabla = 'Pago', TIPOS_PAGO

        # Seleccionar subtipo por probabilidad acumulada
        r, acum, subtipo, media_srv = random.random(), 0, None, 0
        for k, v in tabla.items():
            acum += v['prob']
            if r <= acum:
                subtipo, media_srv = k, v['servicio']
                break

        env.process(cliente_sim(
            env, cliente_id, banco, accion, subtipo,
            media_srv, datos, replica_id, escenario
        ))

# ==============================================================================
# PASO 8: FUNCIÓN DE SIMULACIÓN
# ==============================================================================

def correr_simulacion(nombre_escenario):
    datos_totales = []
    for replica in range(NUM_REPLICAS):
        random.seed(RANDOM_SEED + replica)
        env   = simpy.Environment()
        banco = Banco(env, nombre_escenario)
        env.process(llegada_clientes(env, banco, datos_totales, replica + 1, nombre_escenario))
        env.run(until=TIEMPO_SIM)
    return pd.DataFrame(datos_totales)

# ==============================================================================
# PASO 9: EJECUTAR ESCENARIOS
# ==============================================================================

print("Ejecutando simulación M/M/1 por cajero...\n")

df_3M   = correr_simulacion("3M")
df_1R2P = correr_simulacion("1R-2P")
df_2R1P = correr_simulacion("2R-1P")
df_4M   = correr_simulacion("4M")

df_final = pd.concat([df_3M, df_1R2P, df_2R1P, df_4M], ignore_index=True)

print(f"Total registros: {len(df_final)}")

# ==============================================================================
# PASO 10: RESULTADOS
# ==============================================================================

SEP = "=" * 60

# ------------------------------------------------------------------------------
# PUNTO 1: CAJERO CON MAYOR Y MENOR TIEMPO PROMEDIO DE SERVICIO
# ------------------------------------------------------------------------------

print(f"\n{SEP}")
print("1. TIEMPOS PROMEDIO DE SERVICIO POR CAJERO")
print(SEP)

tiempos_cajero = (
    df_final
    .groupby(['Escenario', 'Cajero_ID'])['Tiempo_Servicio']
    .mean()
    .reset_index()
    .rename(columns={'Tiempo_Servicio': 'Tserv_Prom'})
    .sort_values(['Escenario', 'Tserv_Prom'])
)
print(tiempos_cajero.to_string(index=False))

rapido = tiempos_cajero.loc[tiempos_cajero['Tserv_Prom'].idxmin()]
lento  = tiempos_cajero.loc[tiempos_cajero['Tserv_Prom'].idxmax()]

print(f"\n>> Cajero MÁS RÁPIDO: {rapido['Cajero_ID']} ({rapido['Escenario']}) "
      f"→ {rapido['Tserv_Prom']:.2f} min")
print(f">> Cajero MÁS LENTO:  {lento['Cajero_ID']}  ({lento['Escenario']})  "
      f"→ {lento['Tserv_Prom']:.2f} min")

# ------------------------------------------------------------------------------
# PUNTO 2: PROMEDIO DE USUARIOS POR SUBTIPO (Rápido, Normal, Lento, MuyLento)
#           en la totalidad de cajeros — sin segregar por escenario
# ------------------------------------------------------------------------------

print(f"\n{SEP}")
print("2. PROMEDIO DE USUARIOS POR SUBTIPO (en todos los cajeros)")
print(SEP)

# Contar por réplica y subtipo (todos los escenarios juntos)
usuarios_subtipo = (
    df_final
    .groupby(['Replica', 'Accion', 'Subtipo'])
    .size()
    .reset_index(name='Cantidad')
)

prom_subtipo = (
    usuarios_subtipo
    .groupby(['Accion', 'Subtipo'])['Cantidad']
    .mean()
    .reset_index()
    .rename(columns={'Cantidad': 'Promedio_por_Replica'})
    .sort_values(['Accion', 'Promedio_por_Replica'], ascending=[True, False])
)
print(prom_subtipo.to_string(index=False))

print("\n>> Subtipo con MAYOR promedio de usuarios:")
idx_max = prom_subtipo['Promedio_por_Replica'].idxmax()
fila_max = prom_subtipo.loc[idx_max]
print(f"   {fila_max['Accion']} - {fila_max['Subtipo']}: {fila_max['Promedio_por_Replica']:.1f} usuarios/réplica")

print(">> Subtipo con MENOR promedio de usuarios:")
idx_min = prom_subtipo['Promedio_por_Replica'].idxmin()
fila_min = prom_subtipo.loc[idx_min]
print(f"   {fila_min['Accion']} - {fila_min['Subtipo']}: {fila_min['Promedio_por_Replica']:.1f} usuarios/réplica")

# ------------------------------------------------------------------------------
# PUNTO 3: TOTAL DE USUARIOS POR TIPO EN CADA RÉPLICA
#           + escenario/réplica con MENOR cantidad total
# ------------------------------------------------------------------------------

print(f"\n{SEP}")
print("3. TOTAL DE USUARIOS POR TIPO (Subtipo) EN CADA RÉPLICA")
print(SEP)

# Desglose por Escenario, Réplica y Subtipo
usuarios_tipo_replica = (
    df_final
    .groupby(['Escenario', 'Replica', 'Accion', 'Subtipo'])
    .size()
    .reset_index(name='Total_Usuarios')
    .sort_values(['Escenario', 'Replica', 'Accion', 'Subtipo'])
)
print(usuarios_tipo_replica.to_string(index=False))

# Total general por Escenario y Réplica (para identificar el menor)
total_por_replica = (
    df_final
    .groupby(['Escenario', 'Replica'])
    .size()
    .reset_index(name='Total_Usuarios')
)

menor_rep   = total_por_replica.loc[total_por_replica['Total_Usuarios'].idxmin()]
escenario_menor = menor_rep['Escenario']
replica_menor   = menor_rep['Replica']

print(f"\n>> Réplica con MENOR cantidad de usuarios:")
print(f"   Escenario: {escenario_menor} | Réplica: {replica_menor} "
      f"| Total: {menor_rep['Total_Usuarios']} usuarios")

# Detalle de esa réplica por subtipo
detalle_menor = usuarios_tipo_replica[
    (usuarios_tipo_replica['Escenario'] == escenario_menor) &
    (usuarios_tipo_replica['Replica']   == replica_menor)
]
print("\n   Detalle por tipo:")
print(detalle_menor[['Accion','Subtipo','Total_Usuarios']].to_string(index=False))

# ------------------------------------------------------------------------------
# PUNTO 4: NECESIDAD DE NUEVO CAJERO (criterios completos M/M/1)
# ------------------------------------------------------------------------------

print(f"\n{SEP}")
print("4. NECESIDAD DE NUEVO CAJERO — CRITERIOS DEL MODELO M/M/1")
print(SEP)

num_cajeros = {"3M": 3, "4M": 4, "1R-2P": 3, "2R-1P": 3}
resumen_criterios = []

for esc in ["3M", "1R-2P", "2R-1P", "4M"]:
    df_esc        = df_final[df_final['Escenario'] == esc]
    c             = num_cajeros[esc]
    lambda_total  = len(df_esc) / (NUM_REPLICAS * TIEMPO_SIM)
    lambda_cajero = lambda_total / c
    mu_cajero     = 1 / df_esc['Tiempo_Servicio'].mean()
    rho           = lambda_cajero / mu_cajero
    Wq            = df_esc['Tiempo_Espera'].mean()
    Lq            = lambda_total * Wq
    W             = df_esc['Tiempo_Total'].mean()
    L             = lambda_total * W
    resumen_criterios.append({
        'Escenario': esc, 'c': c,
        'λ_cajero': round(lambda_cajero, 4),
        'μ_cajero': round(mu_cajero, 4),
        'ρ':        round(rho, 4),
        'Wq(min)':  round(Wq, 4),
        'Lq':       round(Lq, 4),
        'W(min)':   round(W, 4),
        'L':        round(L, 4),
        'Estable':  'Sí ✓' if rho < 1 else 'No ✗',
    })

df_criterios = pd.DataFrame(resumen_criterios)
print(df_criterios.to_string(index=False))

# Comparar 3M vs 4M para decidir si se necesita cajero adicional
wq_3M = df_final[df_final['Escenario']=='3M']['Tiempo_Espera'].mean()
wq_4M = df_final[df_final['Escenario']=='4M']['Tiempo_Espera'].mean()
rho_3M = df_criterios.loc[df_criterios['Escenario']=='3M', 'ρ'].values[0]

reduccion = (wq_3M - wq_4M) / wq_3M * 100
print(f"\n>> ρ escenario base (3M):        {rho_3M:.4f}")
print(f">> Wq con 3 cajeros:             {wq_3M:.4f} min")
print(f">> Wq con 4 cajeros:             {wq_4M:.4f} min")
print(f">> Reducción en Wq al agregar 1: {reduccion:.1f}%")

if rho_3M > 0.75:
    print("\n⚠ RECOMENDACIÓN: ρ > 0.75 indica alta carga por cajero.")
    print("  Se RECOMIENDA agregar un 4to cajero para reducir tiempos de espera.")
else:
    print("\n✓ El sistema con 3 cajeros opera con carga aceptable (ρ < 0.75).")
    print(f"  La adición de un 4to cajero reduciría Wq en {reduccion:.1f}%.")
    print("  Evaluar costo-beneficio según política del banco.")

# ------------------------------------------------------------------------------
# PUNTO 5: CONFIGURACIÓN ÓPTIMA — CAJEROS EXCLUSIVOS (especializados)
# El enunciado pide decidir cuántos cajeros deben ser exclusivos
# para pagos y cuántos para retiros → comparar SOLO 1R-2P vs 2R-1P
# ------------------------------------------------------------------------------

print(f"\n{SEP}")
print("5. CONFIGURACIÓN ÓPTIMA: CAJEROS EXCLUSIVOS PARA PAGOS Y RETIROS")
print(SEP)

# Solo escenarios especializados
esp_escenarios = ["1R-2P", "2R-1P"]
espera_esp = df_final[df_final['Escenario'].isin(esp_escenarios)].groupby('Escenario')['Tiempo_Espera'].mean()

print("Comparación entre configuraciones especializadas:")
print(f"  1 Retiro + 2 Pagos (1R-2P): Wq = {espera_esp['1R-2P']:.4f} min")
print(f"  2 Retiros + 1 Pago  (2R-1P): Wq = {espera_esp['2R-1P']:.4f} min")

# Wq por acción en cada escenario especializado
print("\nWq desglosado por tipo de transacción:")
for esc in esp_escenarios:
    df_esc = df_final[df_final['Escenario'] == esc]
    wq_ret = df_esc[df_esc['Accion']=='Retiro']['Tiempo_Espera'].mean()
    wq_pag = df_esc[df_esc['Accion']=='Pago']['Tiempo_Espera'].mean()
    print(f"  {esc}: Retiro={wq_ret:.4f} min  |  Pago={wq_pag:.4f} min")

# Decisión
if espera_esp["2R-1P"] < espera_esp["1R-2P"]:
    rec_esp = "2R-1P"
    print("\n>> RECOMENDACIÓN: 2 cajeros EXCLUSIVOS para Retiros + 1 cajero EXCLUSIVO para Pagos")
    print("   Justificación: el 70% de los clientes hacen retiros → mayor capacidad")
    print("   en retiros reduce el Wq global del sistema.")
else:
    rec_esp = "1R-2P"
    print("\n>> RECOMENDACIÓN: 1 cajero EXCLUSIVO para Retiros + 2 cajeros EXCLUSIVOS para Pagos")
    print("   Justificación: los pagos tienen mayor tiempo de servicio promedio →")
    print("   dedicar más cajeros a pagos reduce la saturación de esa cola.")

mejor_escenario = rec_esp  # Para usar en conclusiones

# ==============================================================================
# VALIDACIÓN ANALÍTICA M/M/1 — Simulado vs Fórmulas Teóricas
# Wq_teo = ρ / (μ(1-ρ))    Lq_teo = ρ² / (1-ρ)
# ==============================================================================

print(f"\n{SEP}")
print("VALIDACIÓN ANALÍTICA M/M/1 (Simulado vs Teórico)")
print(SEP)
print(f"{'Escenario':<10} {'ρ':>6} {'Wq_sim':>8} {'Wq_teo':>8} {'Lq_sim':>8} {'Lq_teo':>8}")
print("-" * 55)

for row in resumen_criterios:
    esc  = row['Escenario']
    rho  = row['ρ']
    mu   = row['μ_cajero']
    Wq_s = row['Wq(min)']
    Lq_s = row['Lq']
    if rho < 1:
        Wq_t = round(rho / (mu * (1 - rho)), 4)
        Lq_t = round(rho**2 / (1 - rho), 4)
    else:
        Wq_t = Lq_t = float('inf')
    print(f"{esc:<10} {rho:>6.4f} {Wq_s:>8.4f} {Wq_t:>8.4f} {Lq_s:>8.4f} {Lq_t:>8.4f}")



# ==============================================================================
# INTERVALOS DE CONFIANZA (95%) - TODOS LOS ESCENARIOS
# ==============================================================================

print(f"\n{SEP}")
print("INTERVALOS DE CONFIANZA AL 95% - Wq por escenario")
print(SEP)

for esc in ["3M", "1R-2P", "2R-1P", "4M"]:
    serie = (
        df_final[df_final['Escenario'] == esc]
        .groupby('Replica')['Tiempo_Espera']
        .mean()
    )
    media = serie.mean()
    desv  = serie.std(ddof=1)
    n     = len(serie)
    err   = t_dist.ppf(0.975, n - 1) * desv / np.sqrt(n)
    print(f"  {esc}: media={media:.4f}  IC=[{media-err:.4f}, {media+err:.4f}]")

# ==============================================================================
# RESUMEN ESTADÍSTICO
# ==============================================================================

print(f"\n{SEP}")
print("RESUMEN ESTADÍSTICO GENERAL")
print(SEP)
print(df_final[['Tiempo_Espera','Tiempo_Servicio','Tiempo_Total']].describe().round(4))

# ==============================================================================
# VISUALIZACIONES
# ==============================================================================

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Simulación Bancaria M/M/1 - Comparación de Escenarios",
             fontsize=14, fontweight='bold')

# Gráfica 1: Wq promedio por escenario
sns.barplot(ax=axes[0,0], data=df_final, x='Escenario', y='Tiempo_Espera',
            hue='Escenario', palette='viridis', errorbar=None, legend=False)
axes[0,0].set_title('Wq Promedio por Escenario')
axes[0,0].set_ylabel('Minutos')

# Gráfica 2: Distribución de esperas (boxplot)
sns.boxplot(ax=axes[0,1], data=df_final, x='Escenario', y='Tiempo_Espera',
            hue='Escenario', palette='rocket', legend=False)
axes[0,1].set_title('Distribución de Esperas')
axes[0,1].set_ylabel('Minutos')

# Gráfica 3: Wq por tipo de acción
sns.barplot(ax=axes[1,0], data=df_final, x='Escenario', y='Tiempo_Espera',
            hue='Accion', palette='magma', errorbar=None)
axes[1,0].set_title('Wq por Tipo de Acción')
axes[1,0].set_ylabel('Minutos')
axes[1,0].legend(title='Acción')

# Gráfica 4: Tiempo de servicio por cajero (escenario 3M como ejemplo M/M/1)
datos_cajero = (
    df_final
    .groupby(['Escenario', 'Cajero_ID'])['Tiempo_Servicio']
    .mean()
    .reset_index()
)
sns.barplot(ax=axes[1,1], data=datos_cajero, x='Cajero_ID', y='Tiempo_Servicio',
            hue='Escenario', palette='crest', errorbar=None)
axes[1,1].set_title('Tiempo Servicio Promedio por Cajero')
axes[1,1].set_ylabel('Minutos')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('graficas_simulacion.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráficas guardadas como 'graficas_simulacion.png'")

# ==============================================================================
# CONCLUSIONES
# ==============================================================================

print(f"\n{SEP}")
print("CONCLUSIONES")
print(SEP)

conclusiones = {
    "3M": """
  El escenario MIXTO de 3 cajeros (M/M/1 c/u) presentó el mejor desempeño.
  Al permitir que cualquier cajero atienda Retiro o Pago, se evita que
  un cajero quede ocioso mientras el otro está saturado.
  Esto reduce el Wq global y confirma la ventaja teórica del sistema mixto.
    """,
    "4M": """
  El escenario con 4 cajeros mixtos (M/M/1 c/u) presentó el mejor desempeño.
  Esto sugiere que el sistema actual con 3 cajeros está cerca de saturación
  y la adición de un cuarto cajero reduce significativamente los tiempos de espera.
    """,
    "2R-1P": """
  El escenario 2 Retiro + 1 Pago fue el mejor entre los especializados.
  El 70% de los clientes realizan retiros, por lo que concentrar
  mayor capacidad en ese servicio reduce el Wq global.
    """,
    "1R-2P": """
  El escenario 1 Retiro + 2 Pago presentó el mejor balance.
  Los pagos demandan más tiempo de servicio, justificando
  dedicar más cajeros a esa operación.
    """,
}
print(conclusiones.get(mejor_escenario, "Revisar métricas para más detalle."))